#### Versions (Delta Versions)
- Every write creates a new table snapshot version in Delta log.
- Tracks state of the whole table at a point in time.
- Used for: Time Travel, Audit/history, Rollback/restore

Example: Version 3 = table after insert.
#### CDC / CDF
- Tracks row-level changes between versions.
- Shows what changed: insert | delete | update_preimage | update_postimage
- Used for incremental pipelines and replication.

Summary 

Version = full snapshot (“what the table looked like”)

CDC = change stream (“what changed between snapshots”)

In [0]:
CREATE SCHEMA IF NOT EXISTS orderdb;
-- schema and database are same


In [0]:

CREATE OR REPLACE TABLE orderdb.products (
    product_id INT,
    product_name STRING,
    category STRING,
    brand STRING,
    price DECIMAL(10,2),
    stock_qty INT,
    last_updated TIMESTAMP
)
USING DELTA;

In [0]:
SHOW TBLPROPERTIES orderdb.products;

-- property delta.enableChangeDataFeed

In [0]:
SHOW TBLPROPERTIES orderdb.products
('delta.enableChangeDataFeed');

In [0]:
-- enable cdc
ALTER TABLE orderdb.products
SET TBLPROPERTIES (
 delta.enableChangeDataFeed = true
);

In [0]:
-- check again

SHOW TBLPROPERTIES orderdb.products
('delta.enableChangeDataFeed');

In [0]:
DESCRIBE HISTORY orderdb.products;

In [0]:
SELECT * FROM table_changes('orderdb.products',1)

In [0]:
-- CDC , change data capture

In [0]:
-- insert one record 
INSERT INTO orderdb.products VALUES
(105,'Headphones','Accessories','Sony',4500,30,current_timestamp());

In [0]:
-- normal table operation
SELECT * 
FROM orderdb.products;

In [0]:
DESCRIBE HISTORY orderdb.products;

In [0]:
SELECT *
FROM table_changes('orderdb.products', 2);
-- scroll output towards right, check _change_type

In [0]:
UPDATE orderdb.products
SET
 price = 4200,
 stock_qty = 25,
 last_updated=current_timestamp()
WHERE product_id=105;

In [0]:
DESCRIBE HISTORY orderdb.products;

In [0]:
SELECT *
FROM table_changes('orderdb.products',3);
-- scroll output towards right, check _change_type

In [0]:
DELETE FROM orderdb.products
WHERE product_id=105;

In [0]:
DESCRIBE HISTORY orderdb.products;

In [0]:
SELECT *
FROM table_changes('orderdb.products',4);
-- scroll output towards right, check _change_type

In [0]:
-- all operations 
SELECT
 product_id,
 product_name,
 price,
 stock_qty,
 _change_type,
 _commit_version
FROM table_changes('orderdb.products',2)
ORDER BY _commit_version;

In [0]:
%python
# CDC as stream

cdc_df = (spark.readStream  # stream  read, not a batch read
  .format("delta")
  .option("readChangeFeed","true")
  .option("startingVersion",3)
  .table("orderdb.products") # change if your table name is different
)

In [0]:
%python
# please terminate this stream , else will run always
display(cdc_df)

In [0]:
%python
# write the changes to another table
# please stop the stream
(cdc_df.writeStream
 .format("delta")
 .outputMode("append")
 .option("checkpointLocation","/tmp/chk/products_cdc")
 .table("orderdb.products_changes")) # change table name, `

In [0]:
SELECT * FROM orderdb.products_changes

- table_changes() → batch query CDC

- readStream + readChangeFeed=true → streaming CDC

#### Patterns
- Log-based CDC: Source DB logs (MySQL binlog, Oracle redo) → Kafka/Debezium → Delta.
- Delta CDF as stream source : table changes → readStream → downstream tables
- Bronze → Silver CDC Propagation: Raw change events in Bronze, dedupe/order them, apply to Silver current-state tables for medallian
- SCD Type 1 Pattern : London -> Paris  (keep only Paris)
- SCD Type 2 Pattern: history preserved, old row expires, new row inserted for Current + historical versions.
- Fan-Out CDC Pattern: One source change feeds multiple consumers.
- CDC Replay Pattern, Use versions + CDF to replay from version N after failures. Good recovery pattern/refill
- Incremental Materialized View Pattern, Use CDC to refresh aggregates incrementally instead of recomputing (MERGE INTO)
